# 📝 Semana 14 · Unidad 4 — Tarea Unidad 4: Binary Search Trees

**Universidad de Talca — Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 4: Diccionarios |
| **Tema** | BST — Delete, Análisis e Implementación |
| **Puntaje total** | 40 puntos |
| **Modalidad** | Individual |

---

## 📋 Instrucciones

1. Completa **todas** las preguntas en este notebook.
2. Tu notebook debe **ejecutar sin errores** de principio a fin.
3. Incluye tus respuestas teóricas directamente en las celdas Markdown correspondientes.
4. **No importes librerías externas** no incluidas en el Setup.
5. Sube el archivo `.ipynb` con tu apellido en el nombre: `02_ALG_ASIG_BST_APELLIDO.ipynb`

---
> ⚠️ **Integridad académica:** Todo el código debe ser tuyo. No se permite copiar soluciones de internet ni de compañeros.

In [ ]:
## Setup (NO MODIFICAR)
import math
import random
import statistics
import sys
import unittest

# Un BST degenerado de 1000 nodos (Pregunta 3) recursa 1000 niveles;
# subimos el límite de recursión de Python, que por defecto es justo 1000.
sys.setrecursionlimit(10_000)

# Identificación del estudiante
NOMBRE  = "Tu nombre aquí"
RUT     = "XX.XXX.XXX-X"
SECCION = "XX"

print(f"Estudiante: {NOMBRE}")
print(f"RUT: {RUT}")
print(f"Sección: {SECCION}")

def _size(node):
    return 0 if node is None else node.size

def altura_bst(root):
    if root is None: return 0
    return 1 + max(altura_bst(root.left), altura_bst(root.right))

def inorden(root):
    result = []
    def _in(node):
        if node is None: return
        _in(node.left); result.append(node.key); _in(node.right)
    _in(root)
    return result

# Gráfico de barras en texto, para la Pregunta 3

def mostrar_barras(filas, unidad="ms", ancho=40, formato="8.2f", maximo=None):
    """
    Imprime un gráfico de barras horizontal en texto.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra que representa a `maximo`.
        formato (str): formato del valor impreso, por ejemplo "8.2f" o "8,.0f".
        maximo (float | None): valor que ocupa todo el ancho; por defecto, el mayor
            de `filas`. Pasar el mismo valor a dos gráficos los deja en la misma escala.
    """
    maximo = maximo or max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:{formato}} {unidad}")


print("✅ Setup listo")

---
## Pregunta 1 — Análisis Teórico del BST (10 puntos)

### 1.1 Árbol de inserción (4 pts)

Dado el siguiente orden de inserción:  
`'F', 'B', 'G', 'A', 'D', 'I', 'C', 'E', 'H'`

**a)** Dibuja el BST resultante (usa texto ASCII o descripción estructurada).  
Muestra la raíz, cada nodo con su hijo izquierdo y derecho, y el campo `size` de cada nodo.

**b)** ¿Cuál es la altura del árbol? ¿Coincide con el valor esperado 2·log₂(9) ≈ 6.3?

**c)** Traza la búsqueda `get('E')` paso a paso: indica los nodos visitados y en qué dirección va en cada uno.

### Tu Respuesta 1.1:
*(edita esta celda)*

---

### 1.2 Peor caso y árbol degenerado (3 pts)

**a)** ¿En qué orden de inserción de las mismas 9 claves `{A,B,C,D,E,F,G,H,I}` se obtiene el árbol de **mayor altura** posible? ¿Cuál sería esa altura?

**b)** Describe la forma que tendría ese árbol. ¿En qué se parece a una estructura de datos vista en unidades anteriores?

**c)** Para ese árbol degenerado, ¿cuántas comparaciones requiere `get('I')` (la última clave insertada en orden ascendente)? Compara con el BST del punto 1.1.

### Tu Respuesta 1.2:
*(edita esta celda)*

---

### 1.3 Comparación de implementaciones (3 pts)

Completa la siguiente tabla con las complejidades **en el caso promedio** (inserción aleatoria para BST):

| Operación | Sequential ST | Binary Search ST | BST (promedio) | BST (peor caso) |
|-----------|:-------------:|:----------------:|:--------------:|:---------------:|
| `get` | | | | |
| `put` | | | | |
| `min/max` | | | | |
| `floor/ceiling` | | | | |
| `rank/select` | | | | |

Luego responde: ¿en qué escenario sería preferible usar Binary Search ST en lugar de BST, a pesar de que BST es más rápido en `put`?

### Tu Respuesta 1.3:
*(edita esta celda)*

---
## Pregunta 2 — Implementación: delete con el algoritmo de Hibbard (20 puntos)

### El problema de eliminar en un BST

Eliminar un nodo de un BST es más complejo que `get` o `put` porque debemos mantener la propiedad BST y el campo `size` en todos los nodos del camino.

Existen tres casos:

**Caso 1 — el nodo es una hoja** (no tiene hijos):
```
Eliminar 'A':      Antes         Después
                     E             E
                    / \           / \
                   A   R    →    None  R
Simplemente retornar None.
```

**Caso 2 — el nodo tiene exactamente un hijo**:
```
Eliminar 'R':      Antes         Después
                     E             E
                    / \           / \
                   A   R    →    A   M
                      /
                     M
Retornar el único hijo: node.right = node.left o node.right.
```

**Caso 3 — el nodo tiene dos hijos** (el algoritmo de Hibbard):
```
Eliminar 'E':      Antes         Después
                     S             S
                    / \           / \
                   E   X   →    H   X
                  / \           / \
                 A   R         A   R
                    /             /
                   H             None
```

El algoritmo de Hibbard para Caso 3:
1. Encontrar el **sucesor** del nodo a eliminar: el mínimo del subárbol derecho.
2. Guardar la referencia al sucesor.
3. Reemplazar el subárbol derecho del sucesor con `deleteMin(node.right)`.
4. El subárbol izquierdo del sucesor queda igual que el del nodo eliminado.
5. El sucesor ocupa el lugar del nodo eliminado.

```python
# Pseudocódigo del Caso 3:
t = node                           # nodo a eliminar
node = _min(t.right)               # sucesor = mínimo del subárbol derecho
node.right = _deleteMin(t.right)   # reconectar: quitar el sucesor de allí
node.left  = t.left                # heredar el subárbol izquierdo
```

### 2.1 Traza de delete a mano (5 pts)

Dado el BST del punto 1.1:
```
        F
       / \
      B   G
     / \   \
    A   D   I
       / \ /
      C  E H
```

**a)** Traza la eliminación de `'D'` (Caso 3):  
- ¿Cuál es el sucesor de `'D'`?  
- ¿Cómo queda el subárbol con raíz `'B'` después de la eliminación?  
- ¿Qué valores de `size` cambian?

**b)** Dibuja el árbol completo después de eliminar `'D'`.

**c)** Ahora elimina `'B'` del árbol resultante. ¿Cuál es el sucesor de `'B'`? Dibuja el árbol final.

### Tu Respuesta 2.1:
*(edita esta celda)*

### 2.2 Implementar deleteMin y delete (15 pts)

Usa el BST base provisto a continuación. Implementa:
1. `deleteMin()` — eliminar el nodo con la clave mínima
2. `delete(key)` — eliminar cualquier clave usando el algoritmo de Hibbard

In [ ]:
## BST base — NO MODIFICAR
class Node:
    def __init__(self, key, value):
        self.key = key; self.value = value
        self.left = None; self.right = None
        self.size = 1

class BST:
    def __init__(self):
        self._root = None
    def size(self): return _size(self._root)
    def isEmpty(self): return self._root is None
    def get(self, key):
        node = self._root
        while node:
            if key < node.key: node = node.left
            elif key > node.key: node = node.right
            else: return node.value
        return None
    def contains(self, key): return self.get(key) is not None
    def put(self, key, value): self._root = self._put(self._root, key, value)
    def _put(self, node, key, value):
        if node is None: return Node(key, value)
        if key < node.key: node.left  = self._put(node.left,  key, value)
        elif key > node.key: node.right = self._put(node.right, key, value)
        else: node.value = value
        node.size = 1 + _size(node.left) + _size(node.right)
        return node
    def min(self): return self._min(self._root).key if self._root else None
    def _min(self, node):
        if node.left is None: return node
        return self._min(node.left)
    def max(self): return self._max(self._root).key if self._root else None
    def _max(self, node):
        if node.right is None: return node
        return self._max(node.right)
    def keys(self):
        r = []
        def _in(n):
            if n is None: return
            _in(n.left); r.append(n.key); _in(n.right)
        _in(self._root); return r
print("✅ BST base cargado")

In [ ]:
# TODO: Implementa DeleteBST extendiendo BST
# MÉTODOS A IMPLEMENTAR:
#   - deleteMin()     → eliminar el nodo con la clave mínima
#   - _deleteMin(node) → versión recursiva interna
#   - delete(key)     → eliminar cualquier clave (3 casos)
#   - _delete(node, key) → versión recursiva interna

class DeleteBST(BST):

    def deleteMin(self):
        """
        Eliminar el nodo con la clave mínima.
        Si el árbol está vacío, no hacer nada.
        """
        if self.isEmpty(): return
        self._root = self._deleteMin(self._root)

    def _deleteMin(self, node):
        """
        Eliminar el nodo mínimo del subárbol con raíz node.
        Retornar la nueva raíz del subárbol.
        HINT:
          Si node.left es None → este es el mínimo → retornar node.right
          Si no → node.left = _deleteMin(node.left)
                  actualizar node.size
                  retornar node
        """
        # TU CÓDIGO AQUÍ
        pass

    def delete(self, key):
        """
        Eliminar la clave key del BST.
        Si key no existe, no hacer nada.
        """
        if key is None: raise ValueError("key no puede ser None")
        self._root = self._delete(self._root, key)

    def _delete(self, node, key):
        """
        Eliminar key del subárbol con raíz node.
        Retornar la nueva raíz del subárbol.

        HINT — estructura del algoritmo:
          1. Navegar hasta el nodo a eliminar (igual que _get)
          2. Caso 1: node.right is None → retornar node.left
          3. Caso 2: node.left  is None → retornar node.right
          4. Caso 3 (Hibbard):
               t = node                         # guardar referencia
               node = _min(t.right)             # sucesor
               node.right = _deleteMin(t.right) # quitar sucesor del subárbol derecho
               node.left  = t.left              # heredar subárbol izquierdo
          5. Actualizar node.size
          6. Retornar node
        """
        # TU CÓDIGO AQUÍ
        pass

In [ ]:
# Pruebas de correctitud — NO MODIFICAR

def check_size(node):
    """True si en todo el subárbol se cumple size = 1 + size(izq) + size(der)."""
    if node is None: return True
    expected = 1 + _size(node.left) + _size(node.right)
    return node.size == expected and check_size(node.left) and check_size(node.right)


class TestDeleteBST(unittest.TestCase):
    """deleteMin y delete (algoritmo de Hibbard) sobre el BST del enunciado."""

    CLAVES = ['F', 'B', 'G', 'A', 'D', 'I', 'C', 'E', 'H']

    def arbol(self):
        bst = DeleteBST()
        for k in self.CLAVES:
            bst.put(k, ord(k))
        return bst

    def test_1_delete_min(self):
        """deleteMin elimina 'A' y deja 8 claves en orden"""
        bst = self.arbol()
        bst.deleteMin()
        self.assertIsNone(bst.get('A'))
        self.assertEqual(bst.keys(), ['B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'])
        self.assertEqual(bst.size(), 8)

    def test_2_delete_min_repetido(self):
        """cinco deleteMin seguidos extraen 1, 3, 4, 5, 7 y vacían el árbol"""
        bst = DeleteBST()
        for k in [5, 3, 7, 1, 4]:
            bst.put(k, k)
        orden = []
        for _ in range(5):
            orden.append(bst.min())
            bst.deleteMin()
        self.assertEqual(orden, [1, 3, 4, 5, 7])
        self.assertTrue(bst.isEmpty())

    def test_3_delete_hoja(self):
        """caso 1: delete('A') elimina una hoja"""
        bst = self.arbol()
        bst.delete('A')
        self.assertIsNone(bst.get('A'))
        self.assertEqual(bst.size(), 8)
        self.assertEqual(bst.keys(), ['B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'])

    def test_4_delete_un_hijo(self):
        """caso 2: delete('G') elimina un nodo con un solo hijo (I)"""
        bst = self.arbol()
        bst.delete('A')
        bst.delete('G')
        self.assertIsNone(bst.get('G'))
        self.assertEqual(bst.size(), 7)
        self.assertEqual(bst.keys(), ['B', 'C', 'D', 'E', 'F', 'H', 'I'])

    def test_5_delete_dos_hijos(self):
        """caso 3 (Hibbard): delete('D') elimina un nodo con dos hijos"""
        bst = self.arbol()
        bst.delete('D')
        self.assertIsNone(bst.get('D'))
        self.assertEqual(bst.keys(), ['A', 'B', 'C', 'E', 'F', 'G', 'H', 'I'])
        self.assertEqual(bst.size(), 8)

    def test_6_delete_raiz(self):
        """delete('F') elimina la raíz"""
        bst = self.arbol()
        bst.delete('D')
        bst.delete('F')
        self.assertIsNone(bst.get('F'))
        self.assertEqual(bst.keys(), ['A', 'B', 'C', 'E', 'G', 'H', 'I'])

    def test_7_delete_inexistente(self):
        """delete de una clave inexistente no cambia size()"""
        bst = self.arbol()
        antes = bst.size()
        bst.delete('Z')
        self.assertEqual(bst.size(), antes)

    def test_8_invariante_size(self):
        """tras 10 deletes aleatorios quedan 10 nodos y el invariante size se cumple"""
        bst = DeleteBST()
        claves = random.sample(range(100), 20)
        for k in claves:
            bst.put(k, k)
        for k in random.sample(claves, 10):
            bst.delete(k)
        self.assertEqual(bst.size(), 10)
        self.assertTrue(check_size(bst._root))

    def test_9_propiedad_bst(self):
        """tras 15 deletes aleatorios quedan exactamente las claves restantes, en orden"""
        bst = DeleteBST()
        datos = random.sample(range(200), 30)
        for k in datos:
            bst.put(k, k)
        eliminar = random.sample(datos, 15)
        for k in eliminar:
            bst.delete(k)
        self.assertEqual(bst.keys(), sorted(k for k in datos if k not in eliminar))


resultado = unittest.main(argv=["ignorado", "TestDeleteBST"], exit=False, verbosity=2)

---
## Pregunta 3 — Análisis Experimental (10 puntos)

### 3.1 Altura promedio vs N (5 pts)

Compara experimentalmente la altura del BST bajo dos condiciones:
- **Aleatorio**: N claves en orden aleatorio
- **Degenerado**: N claves en orden ascendente

Para N ∈ {10, 50, 100, 200, 500, 1000}, genera 20 árboles aleatorios por cada N y promedia la altura.  
Genera un **gráfico de barras en texto** (con `mostrar_barras` del Setup) con las dos alturas y las referencias teóricas (2·log₂N y N) para cada N.

In [ ]:
# TODO: Genera el gráfico de altura promedio vs N
ns = [10, 50, 100, 200, 500, 1000]
REPETICIONES = 20

alturas_random = []   # promedio de 20 ejecuciones por N
alturas_sorted = []   # determinístico (siempre el mismo)

# TU CÓDIGO AQUÍ

# Gráfico en texto: arma filas (etiqueta, valor) para cada N con la altura aleatoria,
# 2·log₂N, la altura degenerada y N, y muéstralas con mostrar_barras
filas = []
# TU CÓDIGO AQUÍ

print("BST — Altura promedio: aleatorio vs degenerado\n")
if filas:
    mostrar_barras(filas, unidad="", formato="7.1f")

# Tabla de resultados
print(f"\n{'n':>6} {'Aleatorio':>12} {'2·log₂n':>10} {'Degenerado':>12} {'N':>6}")
print("-" * 50)
for n, h_rand, h_sorted in zip(ns, alturas_random, alturas_sorted):
    print(f"{n:>6} {h_rand:>12.2f} {2*math.log2(n):>10.2f} {h_sorted:>12.0f} {n:>6}")

### 3.2 Impacto de delete en la altura (5 pts)

El algoritmo de Hibbard tiene un efecto conocido: con muchas operaciones de delete seguidas de insert (mix de operaciones), la altura del árbol puede **aumentar gradualmente** por encima de 2·log₂N.

Diseña y ejecuta el siguiente experimento:
1. Construir un BST con N=500 claves aleatorias.
2. Repetir 1000 veces: eliminar una clave aleatoria del árbol e insertar una nueva clave aleatoria.
3. Medir la altura cada 50 operaciones.
4. Mostrar la altura vs número de operaciones con `mostrar_barras`, indicando la referencia 2·log₂(500).

In [ ]:
# TODO: Implementa el experimento de drift de altura
N = 500
OPERACIONES = 1000
MEDIR_CADA = 50

alturas = []
operaciones_medidas = []

# TU CÓDIGO AQUÍ

# Gráfico en texto: una barra por medición, con la referencia 2·log₂(N)
print(f"Drift de altura tras mezcla de delete/insert (N={N}) — referencia 2·log₂({N}) = {2*math.log2(N):.1f}\n")
if alturas:
    mostrar_barras([(f"{ops} ops", h) for ops, h in zip(operaciones_medidas, alturas)],
                   unidad="", formato="5.0f")

### 3.3 Interpretación (edita esta celda)

**a)** En el gráfico de 3.1: ¿qué tan bien se ajusta la curva aleatoria a la referencia teórica 2·log₂N? ¿Se mantiene ese ajuste para todos los valores de N?

**Respuesta:**

---

**b)** En el experimento de 3.2: ¿la altura del árbol sube, baja o se mantiene estable tras las operaciones de delete/insert? ¿Qué implica esto sobre el algoritmo de Hibbard?

**Respuesta:**

---

**c)** ¿Qué estructura de datos podría resolver el problema del drift de altura? (Solo menciona el nombre — se verá en la próxima unidad.)

**Respuesta:**